In [ ]:
%config Completer.use_jedi=False
%load_ext autoreload
%autoreload 2
from bs4 import BeautifulSoup
import requests
import pathlib
from tqdm.notebook import tqdm
import re
from dateutil import parser
import json
from bs4 import NavigableString

In [ ]:
url = f"https://www.sgutranscripts.org/wiki/SGU_Episode_{{ep}}"
url_5x5 = f"https://www.sgutranscripts.org/wiki/5X5_Episode_{{ep}}"

#### Get the raw html and store them to disk

In [ ]:
for ep in tqdm(list(range(1,973))):
    html = requests.get(url.format(ep=ep))
    path = pathlib.Path(f"./ep_files/SGU_Episode_{ep}.html")
    path.write_text(html.text, encoding="utf-8")

In [ ]:
# ep = 85
# path = pathlib.Path(f"./ep_files/SGU_Episode_{ep}.html")
# html = path.read_text(encoding="utf-8")
# soup = BeautifulSoup(html, "lxml")

#### Define some utilities for parsing the transctipts

In [ ]:
def get_rogues_table(soup):
    """Find the table with the list of Rogues in each episode

    Args:
        soup: The BeautifulSoup object representing the transcript html page.

    Returns:
        The BeautifulSoup tag for the rougues table
    """
    
    def find_rogues_table(tag):
        if tag.name == "table" and "Skeptical Rogues" in list(tag.stripped_strings):
            return True
        return False
    t = soup.find(find_rogues_table)
    return t
    
def get_rogues_table_string(soup):
    """Return the rogues table as a string

    Args:
        soup: The BeautifulSoup object representing the transcript html page.

    Returns:
        The rougues table as a string
    """
    
    s = get_rogues_table(soup).get_text()
    return s

def parse_rogues_table(soup, correction={}):
    """Find and parse the Rogues Table from the transcript html page into a dict.

    Args:
        soup: The BeautifulSoup object representing the transcript html page.
        correction: a dict with manual corrections of transcription errors for this transcript.

    Returns:
        A dict with the transcript metadata extracted from the Rogues table.
    """

    # get the Rogues table tag
    table = get_rogues_table(soup)
    if table is None:
        return table

    # Get the first row and extract the Title from it.
    tr = table.tr
    title = tr.get_text().strip()

    # The next row should be the transcript date.
    tr = tr.find_next_sibling("tr")
    date = " ".join(tr.stripped_strings).replace("Transcript Verified", "").strip()
    date = " ".join(date.split()[0:3])

    # Try to parse the date string into a date and its constituents
    try:
        date = parser.parse(date)
        year = date.date().year
        month = date.date().month
        day = date.date().day
        date = date.strftime("%B %-d %Y")
    except parser.ParserError as e:
        year = ""
        month = ""
        day = ""
        date = ""

    # Find the Skeptical Rogues section of the table and extract their names and abbreviations to a list of strings.
    t = table.find(string=re.compile("^Skeptical Rogues"))
    rogues = []
    for tr in t.find_all_next("tr"):
        if re.search("^[A-Z][a-zA-Z]{0,2}: \S+", tr.get_text().strip()):
            rogues.append(tr.get_text().strip())
        else:
            break

    # Include any manual corrections to the rogues in this transcript.
    if "rogues" in correction:
        rogues.extend(correction["rogues"])
    
    # Create a dict (rogues_map( with the abbreviation as the key and the name as the value
    # Corrections will override previous erroneous entries with the same key.
    rogues_map = {r.split(": ")[0]: r.split(": ")[1].split(",", 1)[0] for r in rogues}

    # The list of names with duplicates removed
    rogues = list(set(rogues_map.values()))

    # Add entries for the Voiceover speaker in the Rogue dict
    rogues_map["Voiceover"] = "Voiceover"
    rogues_map["VO"] = "Voiceover"
    
    # Get the manual corrections for the guest list for this transcript
    correction_guests = []
    if "guests" in correction:
        correction_guests = correction["guests"]

    # Find the list of guest table row tags in the Rogues Table. 
    t_tmp = table.find(string=re.compile("^Guest"))
    guests = []
    if t_tmp:    
        t = t_tmp
        for tr in t.find_all_next("tr"):
            if re.search("^[A-Z][a-zA-Z]{0,2}: \S+", tr.get_text().strip()):
                guests.append(tr)
            else:
                break

    # Sometimes guests have URLs provided so extract those if present.
    # Then get the guest as a string.
    guest_urls = []
    if guests:
        guest_urls = [str(tr.a["href"]) if tr.a else None for tr in guests]
        guests = [tr.get_text().strip() for tr in guests]
    
    # Include the manual guest corrections.
    guests = guests + correction_guests

    # Create a dict (guests_map) of guests where the abbreviation is the key and the value is the name
    # Also create a list of guest dicts (guests)
    # Combine guests_map in to rogues_map so we have one dict of all potential speakers
    if guests:
        guests_map = {r.split(": ")[0].upper(): r.split(": ")[1] for r in guests}
        guests = [{"guest": g.split(",", 1)[0], "guest_title": g.split(",", 1)[1] if len(g.split(",")) > 1 else ""} for g in guests_map.values()]
        for i, g in enumerate(guests):
            if i < len(guest_urls):
                g["guest_url"] = guest_urls[i]
        rogues_map.update({k: v.split(",")[0] for k, v in guests_map.items()}) 

    # Start compiling the return value which will be a dict of metadata from the Rogues Table.
    ret =  {"title": title, "date": date, "year": year, "month": month, "day": day, "rogues": rogues, "rogues_map": rogues_map}

    # Find the Quote of the Week in the Rogues Table.
    t = table.find(string=re.compile("^Quote of the Week"))

    # Extract the Quote, the Quotee, and their URL and add them to the metadata dict.
    if t:
        t = t.find_next("tr")
        quote = " ".join(t.strings) #t.get_text().strip()
        t = t.find_next("tr")
        quotee = " ".join(t.strings) #t.get_text().strip()
        quotee_title = quotee.split(",", 1)[1] if len(quotee.split(",", 1)) > 1 else ""
        quotee = quotee.split(",", 1)[0]
        if t.a:
            ret["quotee_url"] = t.a["href"]
        ret["quote"] = quote
        ret["quotee"] = quotee
        ret["quotee_title"] = quotee_title

    # Add the list of guests to the meta data dict
    if guests:
        ret["guests"] = guests


    return ret


In [ ]:
def parse_toc(soup):
    """Find and parse the Table of Contents to get a list of section headers.

    Args:
        soup: The BeautifulSoup object representing the transcript html page.

    Returns:
        The list of section header tags.
    """
    toc = soup.find(name="div", id="toc")
    if toc:
        toc_links = toc("a")
        return toc_links
    else:
        return []


In [ ]:
def parse_categories(soup):
    """Find and parse the list of Categories to which thie transcript belongs.

    Args:
        soup: The BeautifulSoup object representing the transcript html page.

    Returns:
        The list of Category names.
    """
    cats = []
    l = soup.find("div", id="mw-normal-catlinks")
    if l:
        cat_links = l.find_all("li")
        cats = [l.get_text() for l in cat_links] 
    return cats

In [ ]:
def get_title(soup):
    """Get the Episode Title from the html

    Args:
        soup: The BeautifulSoup object representing the transcript html page.

    Returns:
        The string title.
    """
    s = soup.find(class_="mw-page-title-main")
    if s:
        s = str(s.string)
        return s
    else:
        return None

#### The Document object that can be submitted to the Vectara Index/Upload API is recursive  
#### You can have sections and subsections.
#### The *build_sections* function and its internal *build_subsection* function attempt to recursively
#### build up a document based on the hierarchical structure of the transcripts such that the lowest unit if the utterance of one person.

In [ ]:
from pprint import pprint
template_strings = {
    "Fact/Description",
    "[url_for_TIL publication: title]",
    "_text_when_ROGUE_introduces_new_predictions_",
    "_text_when_ROGUE_references_previous_predictions_",
    "[url_from_news_item_show_notes publication: title]",
    "[url_from_SoF_show_notes publication: title]",
    "S: The Skeptics' Guide to the Universe is produced",
    "S: Skeptics' Guide to the Universe is produced",
    "The Skeptics' Guide to the Universe is produced",
    "Voice-over: The Skeptics' Guide to the Universe is produced",
    "Voiceover: The Skeptics' Guide to the Universe is produced",
}
def build_sections(soup, rogues_map):
    """Build the section list of the Vectara document for a transcript.

    Note this code is very 'hacky' and depends on the HTML format of the transcripts being consistent
        which is not always the case.  Caveat Emptor.

    It is still a work in progress.

    Args:
        soup: The BeautifulSoup object representing the transcript html page.
        rogues_map: The full dict of speakers (rogues and guests) with the keys as their abbreviations and
            the values as their names.

    Returns:
        A list of section dicts (each potentially with subsections) as per the Vectara API spec.
    """
    section_id = 0
    sections = []
    def build_subsection(tag, stop_tags, section_id_dict = {"section_id": 0}):
        #print("A:", tag, stop_tags)
        section = {}
        subsection_list = []
        section["id"] = section_id_dict["section_id"]
        section_id_dict["section_id"] += 1
        metadata = {}
        links = tag.find_all("a", class_="external")
        if links:
            metadata["links"] = [l["href"] for l in links]
        section["text"] = ""
        stop = False
        # if tag.name == "h2":
        #     import pdb
        #     pdb.set_trace()
        speech = False
        if tag.name == "h2" or tag.name == "h3": 
            title = str(list(tag.find("span", class_="mw-headline").strings)[0].strip())
            # if title == "Vocabulary":
            #     import pdb
            #     pdb.set_trace()
            if tag.small:
                timestamp = tag.small.get_text().strip().strip("()")
                hour = 0
                minute = 0
                second = 0
                timesplit = timestamp.split(":")
                if len(timesplit) == 3:
                    hour = timesplit[0]
                    minute = timesplit[1]
                    second = timesplit[2]
                if len(timesplit) == 2:
                    minute = timesplit[0]
                    second = timesplit[1]
                    timestamp = f"0:{minute.zfill(2)}:{second}"
                metadata["timestamp"] = timestamp
                metadata["hour"] = hour
                metadata["minute"] = minute
                metadata["second"] = second
            section["title"] = title
        elif tag.name == "p" or tag.name == "blockquote":
            if tag.big:
                # import pdb
                # pdb.set_trace()
                title = str(tag.big.string)
                section["title"] = title
                stop = "big"
            elif tag.b and tag.contents[0].name == "b" and len(tag.b.get_text()) < 4:
                speaker = str(tag.b.string).upper().strip()
                if ":" in speaker:
                    speaker = speaker[:speaker.find(":")]
                if speaker in rogues_map:
                    metadata["speaker_abbreviation"] = speaker
                    metadata["speaker"] = rogues_map[speaker]
                elif speaker in set(rogues_map.values()):
                    metadata["speaker"] = speaker
                else:
                    if speaker == "":
                        metadata["speaker"] = "<unknown>"
                    elif "/" in speaker:
                        speaker1 = speaker.split("/")[0]
                        speaker2 = speaker.split("/")[1]
                        if speaker1 in rogues_map and speaker2 in rogues_map:
                            metadata["speaker_abbreviation"] = speaker
                            metadata["speaker"] = f"{rogues_map[speaker1]} / {rogues_map[speaker2]}"
                        else:
                            metadata["speaker_abbreviation"] = speaker
                            metadata["speaker"] = speaker
                            #raise ValueError("Invalid Speaker: " + speaker)
                    else:
                        metadata["speaker_abbreviation"] = speaker
                        metadata["speaker"] = speaker
                        #raise ValueError("Invalid Speaker: " + speaker)
                section["text"] = "".join([str(s) for s in list(tag.strings)[1:]]).strip()
                if metadata:
                    section["metadata"] = metadata #json.dumps(metadata)
                return section, tag
            elif re.search("^[A-Z][a-zA-Z]{0,2}:", tag.get_text().lstrip()) or re.search(r"^[A-Z][a-zA-Z]{0,2}  (?:\d:)?\d{1,2}:\d{2} ", tag.get_text().lstrip()):
                t = tag.get_text().strip()
                splits = t.split(maxsplit=1)
                s1 = splits[0]
                s2 = splits[1] if len(splits) > 1 else ""
                if ":" in s1:
                    splits = t.split(": ", 1)
                    speaker = splits[0]
                    text = splits[1] if len(splits) > 1 else ""
                else:
                    speaker = s1
                    splits = s2.split(maxsplit=1)
                    timestamp = splits[0]
                    text = splits[1] if len(splits) > 1 else ""
                    metadata["timestamp"] = timestamp
                speaker = speaker.upper().strip()
                if speaker in rogues_map:
                    metadata["speaker_abbreviation"] = speaker
                    metadata["speaker"] = rogues_map[speaker]
                elif speaker in set(rogues_map.values()):
                    metadata["speaker"] = speaker
                else:
                    if speaker == "":
                        metadata["speaker"] = "<unknown>"
                    elif "/" in speaker:
                        speaker1 = speaker.split("/")[0]
                        speaker2 = speaker.split("/")[1]
                        if speaker1 in rogues_map and speaker2 in rogues_map:
                            metadata["speaker_abbreviation"] = speaker
                            metadata["speaker"] = f"{rogues_map[speaker1]} / {rogues_map[speaker2]}"
                        else:
                            metadata["speaker_abbreviation"] = speaker
                            metadata["speaker"] = speaker
                            #raise ValueError("Invalid Speaker: " + speaker)
                    else:
                        metadata["speaker_abbreviation"] = speaker
                        metadata["speaker"] = speaker
                        #raise ValueError("Invalid Speaker: " + speaker)
                section["text"] = text
                if metadata:
                    section["metadata"] = metadata #json.dumps(metadata)
                return section, tag
            elif re.search(r"^[A-Z][a-zA-Z]{0,2}  (?:\d:)?\d{1,2}:\d{2} ", tag.get_text()):
                raise ValueError("Timestamp in sentence")
            elif tag.name == "blockquote":
                metadata["speaker"] = "<unknown>"
                section["text"] = tag.get_text().strip()
                if metadata:
                    section["metadata"] = metadata #json.dumps(metadata)
                return section, tag
                

            
            
        
        try:
            tag = tag.next_sibling
        except:
            print(tag)
            raise

        
        first = True
        while (tag and tag.name not in stop_tags) and not (stop and tag.find(stop)):
            #print("B:", repr(tag), first, stop_tags, speech)
            if tag.name == "h3":
                subsection, tag = build_subsection(tag, ["h2", "h3"], section_id_dict)
                subsection_list.append(subsection)
                speech = False
            elif tag.name == "p" or tag.name == "blockquote":
                # if tag.name == "blockquote" or tag.get_text().strip() == "S:":
                #     import pdb
                #     pdb.set_trace()
                if tag.big:
                    # import pdb
                    # pdb.set_trace()                    
                    subsection, tag = build_subsection(tag, ["h2", "h3"], section_id_dict)
                    subsection_list.append(subsection)
                    speech = False
                elif tag.name == "blockquote":
                    # import pdb
                    # pdb.set_trace()                    
                    subsection, tag = build_subsection(tag, ["h2", "h3"], section_id_dict)
                    subsection_list.append(subsection)
                    #speech = False
                elif (tag.b and tag.contents[0].name == "b" and len(tag.b.get_text()) < 4) or re.search("^[A-Z][a-zA-Z]{0,2}:", tag.get_text().lstrip()) or re.search(r"^[A-Z][a-zA-Z]{0,2}  (?:\d:)?\d{1,2}:\d{2} ", tag.get_text().lstrip()):
                    subsection, tag = build_subsection(tag, ["h2", "h3"], section_id_dict)
                    subsection_list.append(subsection)
                    speech = True
                    speaker = subsection["metadata"].get("speaker_abbreviation", "")
                else:
                    text = tag.get_text()
                    if text.strip():
                        if speech:
                            text = next(tag.strings)
                            text.replace_with(NavigableString(speaker + ": " + text))
                            subsection, tag = build_subsection(tag, ["h2", "h3"], section_id_dict)
                            subsection_list.append(subsection)
                        else:
                            if all((not text.startswith(ts)) for ts in template_strings): #("_text_when_ROGUE_introduces_new_predictions_" not in text) and ("_text_when_ROGUE_references_previous_predictions_" not in text):
                                section["text"] = (section["text"] + "\n" + text) if "text" in section and section["text"] else text
                                speech = False
            elif first and (tag.name == "ol" or tag.name == "ul" or tag.name == "a" or tag.name == "blockquote" or (tag.name == "div" and (tag.ol or tag.ul))):
                links = tag.find_all("a", class_="external")
                if links:
                    metadata["links"] = metadata.get("links", []) + [l["href"] for l in links]
                text = tag.get_text() 
                if all((not text.startswith(ts)) for ts in template_strings): #("_text_when_ROGUE_introduces_new_predictions_" not in text) and ("_text_when_ROGUE_references_previous_predictions_" not in text):
                    section["text"] = (section["text"] + "\n" + text) if "text" in section and section["text"] else text
                speech = False
            elif str(tag.string) == "\n":
                pass
            else:
                speech = False
            
            if isinstance(tag, NavigableString) and str(tag).strip():
                #print("E:", repr(tag) != "\n", repr(tag), first)
                first = False
            if tag:
                tag = tag.next_sibling
            
        if "title" in section:
            metadata["title"] = section["title"]
        if metadata:
            section["metadata"] = metadata #json.dumps(metadata)

        section["sections"] = subsection_list

        if tag:
            tag = tag.previous_sibling
        return section, tag
        
    toc = parse_toc(soup)
    for i, link in enumerate(toc):
        #print(i, link)
        id = link["href"][1:]
        header = soup.find(id=id).find_parent("h2")
        if not header:
            continue
        else:
            subsection, tag = build_subsection(header, ["h2"])
            sections.append(subsection)


    return sections

#### As I tried to parse the full set of transcripts, I encountered mismatches between the actuals speakers and the speakers in the Rogues Table.
#### *corrections* is my dict of manual corrections.  The key is the episode numbers and the values are list of rogues or guests to be added or overwritten.

In [ ]:
corrections = {
    24: {
        "rogues": [
            "J: James Randi"
        ]
    },
    40: {
        "rogues": [
            "G: Brian Trent"
        ]
    },
    51: {
        "rogues": [
            "P: Perry DeAngelis"
        ],
        "guests": [
            "NA: Neal Adams"
        ]
    },
    52: {
        "guests": [
            "BB: Bill Bennetta"
        ]
    },
    56: {
        "guests": [
            "KF: Ken Feder"
        ]
    },
    57: {
        "rogues": [
            "P: Perry DeAngelis"
        ],
        "guests": [
            "LS: Larry Sarner"
        ]
    },
    58: {
        "guests": [
            "KA: Kimball Atwood, MD"
        ]
    },
    61: {
        "guests": [
            "PP: Phil Plait"
        ]
    },
    63: {
        "guests": [
            "JR: James Randi"
        ]
    },
    64: {
        "guests": [
            "JR: James Randi"
        ]
    },
    65: {
        "guests": [
            "JR: James Randi"
        ]
    },
    66: {
        "guests": [
            "JR: James Randi"
        ]
    },
    67: {
        "guests": [
            "JR: James Randi"
        ]
    },
    69: {
        "guests": [
            "JR: James Randi"
        ]
    },
    70: {
        "guests": [
            "JR: James Randi"
        ]
    },
    71: {
        "guests": [
            "JR: James Randi"
        ]
    },
    72: {
        "guests": [
            "JR: James Randi",
            "RS: Richard Saunders"
        ]
    },
    73: {
        "guests": [
            "JR: James Randi"
        ]
    },
    74: {
        "guests": [
            "JR: James Randi"
        ]
    },
    75: {
        "rogues": [
            "P: Perry DeAngelis"
        ]
    },
    76: {
        "guests": [
            "JR: James Randi"
        ]
    },
    77: {
        "guests": [
            "JR: James Randi"
        ]
    },
    78: {
        "guests": [
            "JR: James Randi"
        ]
    },
    80: {
        "rogues": [
            "P: Perry DeAngelis"
        ],
        "guests": [
            "JR: John Rennie",
            "T: Teller",
            "JU: Jim Underdown"
        ]
    },
    81: {
        "rogues": [
            "P: Perry DeAngelis"
        ],
        "guests": [
            "PP: Phil Plait",
            "AS: Adam Savage",
            "TB:  Tory Belleci"
        ]
    },
    82: {
        "rogues": [
            "P: Perry DeAngelis"
        ],
        "guests": [
            "CH: Christopher Hitchens",
            "MS: Matt Stone"
        ]
    },
    83: {
        "rogues": [
            "P: Perry DeAngelis"
        ],
        "guests": [
            "JS: Julia Sweeney",
            "RW: Richard Wiseman",
            "SD: Scott Dickers"
        ]
    },
    84: {
        "guests": [
            "KF: Ken Feder"
        ]
    },
    87: {
        "guests": [
            "RL: Robert Lancaster"
        ]
    },
    91: {
        "rogues": [
            "P: Perry DeAngelis"
        ]
    },
    104: {
        "guests": [
            "BT: Briant Trent"
        ]
    },
    106: {
        "guests": [
            "JR: James Randi"
        ]
    },
    107: {
        "guests": [
            "JR: James Randi"
        ]
    },
    112: {
        "guests": [
            "JR: James Randi"
        ]
    },
    114: {
        "guests": [
            "JR: James Randi"
        ]
    },
    117: {
        "guests": [
            "JR: James Randi"
        ]
    },
    218: {
        "guests": [
            "M: Michael Vassar"
        ]
    },
    822: {
        "guests": [
            "IL: Iszi Lawrence"
        ]
    },
    373: {
        "guests": [
            "GH: George Hrab"
        ]
    },
    374: {
        "rogues": [
            "BB: Bob Novella"
        ]
    },
    381: {
        "guests": [
            "JN: Joe Nickell"
        ]
    },
    384: {
        "guests": [
            "R: Richard Wiseman"
        ]
    },
    409: {
        "guests": [
            "H: Heather Berlin"
        ]
    },
    414: {
        "guests": [
            "D: Daniel Loxton"
        ]
    },
    423: {
        "guests": [
            "JR: John Rennie"
        ]
    },
    430: {
        "guests": [
            "MK: Marty Klein"
        ]
    },
    434: {
        "guests": [
            "SG: Indre Viskontas"
        ]
    },
    451: {
        "guests": [
            "MK: Michio Kaku"
        ]
    },
    454: {
        "rogues": [
            "BB: Bob Novella"
        ]
    },
    458: {
        "guests": [
            "P: Paul Offit"
        ]
    },
    471: {
        "guests": [
            "D: Doctor A.J. O'Podden"
        ]
    },
    492: {
        "guests": [
            "RS: Richard S."
        ]
    },
    494: {
        "guests": [
            "P: Perry DeAngelis (recording)"
        ]
    },
    502: {
        "guests": [
            "AC: Anderson Cooper (recording)",
            "DB: Dan Burton (recording)"
        ]
    },
    512: {
        "rogues": [
            "R: Rebecca Watson"
        ]
    },
    513: {
        "guests": [
            "JB: Julia Galef"
        ]
    },
    556: {
        "rogues": [
            "GH: George Hrab"
        ]
    },
    568: {
        "guests": [
            "R: Richard Wiseman",
            "RS: Richard Wiseman"
        ]
    },
    581: {
        "rogues": [
            "BOB: Bob Novella"
        ]
    },
    596: {
        "guests": [
            "ML: Marty Klein"
        ]
    },
    690: {
        "guests": [
            "G: George Hrab"
        ]
    },
    761: {
        "guests": [
            "G: George Hrab"
        ]
    },
    798: {
        "guests": [
            "RW: Richard Wiseman"
        ]
    },
    799: {
        "guests": [
            "JR: James Randi (recorded)"
        ]
    },
    810: {
        "guests": [
            "PM: Phil McAlister"
        ]
    },
    813: {
        "rogues": [
            "B: Bob Novella",
            "C: Cara Santa Maria",
            "J: Jay Novella",
            "E: Evan Bernstein"
        ]
    },
    816: {
        "guests": [
            "PG: Philip Goff"
        ]
    },
    824: {
        "rogues": [
            "B: Bob Novella",
            "C: Cara Santa Maria",
            "J: Jay Novella",
            "E: Evan Bernstein"
        ]
    },
    825: {
        "rogues": [
            "B: Bob Novella",
            "C: Cara Santa Maria",
            "J: Jay Novella",
            "E: Evan Bernstein"
        ]
    },
    826: {
        "rogues": [
            "B: Bob Novella",
            "C: Cara Santa Maria",
            "J: Jay Novella",
            "E: Evan Bernstein"
        ]
    },
    849: {
        "guests": [
            "CG: Craig Good"
        ]
    },
    885: {
        "rogues": [
            "E.: Evan Bernstein"
        ]
    },
    898: {
        "guests": [
            "D: David Almeida, SGU Patron"
        ]
    },
    899: {
        "rogues": [
            "S': Steven Novella"
        ]
    },
    904: {
        "rogues": [
            "MA: Ajia Mae Moon"
        ]
    },
    911: {
        "rogues": [
            "GH: George Hrab"
        ]
    },
    927:
    {
        "guests": [
            "PB: Jon Bornstein"
        ]
    },
    933: {
        "rogues": [
            "IC: Ian C."
        ]
    },
    961: {
        "rogues": [
            "E: Jay Novella"
        ]
    },
        
}
            

#### Below I listed out the empty, incomplete, and ok (erroneous but passable) transcripts.

In [ ]:
empties = {
    "105"
}
incompletes = {
    645, 647, 649, 651, 660, 661, 663, 664, 687, 743, 745, 746, 747, 748, 749, 751, 752, 753, 754, 755, 756, 757, 758, 760, 763, 766, 767, 771, 772, 773, 775, 783, 788, 791, 792,
    809, 811, 812, 814, 815, 817, 820, 827, 828, 830, 832, 833, 834, 835, 836, 844
}
ok = {
    39, 54, 380, 411, 597, 652, 656, 797,
    847, 848, 949, 971,
}

#### The create_document_from_file function takes the html file and parses it using the utilities above into a Vectara object

In [ ]:
def create_document_from_file(filename):
    """Create a Vectara document for indexing/upload from the raw html transcript.

    Args:
        filename: the path on disk to the transscript file in raw html format.

    Returns:
        The dict representing the Vectorara document object for the transcript.
    """
    path = pathlib.Path(filename)
    html = path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "lxml")
    
    
    doc = {}
    
    doc["id"] = filename.stem
    doc["title"] = get_title(soup)
    doc["type"] = "structured"
    episode_number = filename.stem.split("_")[2]
    doc["description"] = f"The Skeptic's Guide to the Universe, Episode {episode_number} Transcript"
    if episode_number in empties:
        return doc
    ep = int(episode_number)
    correction = corrections.get(ep, {})
    metadata = parse_rogues_table(soup, correction)
    if metadata is None:
        return None
    rogues_map = metadata.pop("rogues_map")
    
    metadata["categories"] = parse_categories(soup)
    metadata["podcast_url"] = f"https://www.theskepticsguide.org/podcasts/episode-{ep}"
    metadata["episode_number"] = ep
    doc["metadata"] = metadata #json.dumps(metadata)

    doc["sections"] =  build_sections(soup, rogues_map)
    
    return doc

#### Below we loop through all the html transcripts in the download folder, parse them, and write out json files for the resulting Vectara objects.

In [ ]:
import json
for i in range(1, 1000):
    path = pathlib.Path(f"./ep_files/SGU_Episode_{i}.html")
    if not path.exists():
        continue
    html = path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "lxml")
    try:
        doc = create_document_from_file(path)
        if doc is None:
            print(f"Error: {str(path)} is Empty")
            continue
        json_path = pathlib.Path("./ep_json") / path.name
        json_path = json_path.with_suffix(".json")
        with json_path.open("w") as f:
            json.dump(doc, f)
    except Exception as e:
        print(f"{i}: {str(path)}")
        print(e)
        print()
        
                         

### Wikipedia Page
#### After initial feedback from Chase, I decided to manually parse and save create a json for the Wikipedia Page so some basic questions about the SGU could be answered.

In [ ]:
wiki = {
    "id": "SGU_Wikipedia_Summary",
    "type": "structured",
    "title": "The Skeptics' Guide to the Universe",
    "description": "An explanation of what 'The Skeptics' Guide to the Universe' (SGU) is.",
    "metadata" : {
        "source_url": "https://en.wikipedia.org/wiki/The_Skeptics%27_Guide_to_the_Universe"
    },
    "sections": [
        {
            "id": 0,
            "title": "What is 'The Skeptics' Guide to the Universe (SGU)'",
            "metadata": {
            },
            "text": """The Skeptics' Guide to the Universe (SGU) is an American weekly skeptical podcast hosted by Steven Novella, MD, along with a panel of contributors. The official podcast of the New England Skeptical Society, it was named to evoke The Hitchhiker's Guide to the Galaxy. The show features discussions of recent scientific developments in layman's terms, and interviews authors, people in the area of science, and other famous skeptics. The SGU podcast includes discussions of myths, conspiracy theories, pseudoscience, the paranormal, and other forms of superstition, from the point of view of scientific skepticism.""",
        },
        {
            "id": 1,
            "title": "Hosts",
            "metadata": {
            },
            "text": """The SGU podcast was first released in May 2005. The original lineup of the podcast consisted of the Novella brothers, Steven Novella, Robert "Bob" Novella and Jay Novella, along with Evan Bernstein, and Perry DeAngelis. DeAngelis contributed until his death in 2007, shortly before his 44th birthday. Rebecca Watson joined in 2006 and later left the podcast in 2014. Cara Santa Maria joined the cast in July 2015.

The podcast is affiliated with the New England Skeptical Society and with the SGU Productions LLC.

Until 2018 there were no full-time employees of the podcast, although that was in part due[according to whom?] to a defamation lawsuit filed by Edward Tobinick that consumed financial resources that would have otherwise been available.[3] In 2018 Jay Novella, who had previously both been one of the cast and managed the podcast's website, was employed full-time as a result of reaching a Patreon target of 3,000 donors.""",
        },
        {
            "id": 2,
            "title": "Production",
            "metadata": {
            },
            "text": """Segments include interviews, discussions of significant but largely unknown figures in science, short games and puzzles played with the audience or between the panellists, and accounts of relevant events in the news that relate to skepticism.[citation needed] Shows last about 80 minutes, although on September 23, 2011, SGU produced a 24-hour-long podcast with contributions by skeptics from around the world. It was referred to as SGU-24.""",
        },
        {
            "id": 3,
            "title": "Theme Music",
            "metadata": {
            },
            "text": """The show's theme music is "Theorem" by the San Francisco rock band, Kineto. The theme was acquired from the Podsafe Music Network. Prior to the November 2, 2005 show, Thomas Dolby's "She Blinded Me with Science" was the show's theme.""",
        },
        {
            "id": 4,
            "title": "Guests",
            "metadata": {
            },
            "text": """Many Skeptics' Guide episodes contain interviews. Often the interviews feature well-known scientists or skeptics, for instance Massimo Pigliucci or Joe Nickell. Rarely the guests are proponents of fringe or pseudoscientific views. Some episodes have guest rogues, such as Bill Nye, participating in the entire podcast.""",
        },
        {
            "id": 5,
            "title": "Recognition",
            "metadata": {
            },
            "text": """The Skeptics' Guide won the 2009 Podcast Awards in the "Education" category, and the 2010, 2011, 2012, and 2014 Podcast Awards in the "Science" category.

It was also a 2014 "Dose of Rationality" Top 10 Podcast, and a 2010 Physics.org Best Podcast nominee."""
        },
        {
            "id": 6,
            "title": "Sponsors and Membership",
            "metadata": {
            },
            "text": """On July 30, 2013, Steven Novella announced that the SGU would begin offering membership and airing sponsors. Novella went on to say that the money raised would go into funding skeptical activities, including but not limited to, development of skeptical educational content and web-series such as "Occ The Skeptical Caveman". The addition of sponsors is not permanent, according to Novella, they shall be removed "if 4% of listeners support the SGU through membership at an average of the $8 per month level." Though membership has begun, the SGU continues to publish a free weekly sponsored podcast. Membership entitles one to an ad-free version of The SGU, extra content, and discounts to NECSS (The Northeast Conference on Science and Skepticism). Membership range from $4/month to $200/month.[20]

Additional financial support from listeners is provided through Patreon. The SGU has established several goals after achieving a certain number of financial supporters. A major benchmark was reached in 2018 with 3,000 Patreon supporters that sustained enough predictable income for a full-time employee.[3] Other benchmarks include a 12-hour and 24-hour live show after reaching 4,000 and 5,500 supporters, respectively. These live shows may be located on the most complete and accurate reproduction of the Starship Enterprise Star Trek: The Original Series set, which was built by James Cawley and can be seen on the SGU Patreon page introduction video."""
        },
        {
            "id": 7,
            "title": "The Skeptics' Guide to the Universe book",
            "metadata": {
            },
            "text": """The Skeptics' Guide to the Universe: How to Know What's Really Real in a World Increasingly Full of Fake is a 2018 book written by Steven Novella and co-authored by the other current co-hosts of The Skeptics' Guide to the Universe podcast—Bob Novella, Cara Santa Maria, Jay Novella, and Evan Bernstein. It also contains posthumous material from former co-host Perry DeAngelis. The book is meant to be an all-encompassing guide to skeptical thinking. In an interview with The European Skeptics Podcast, Jay Novella describes their approach to writing the book from the "point of view of an alien species observing the earth from a skeptical perspective using critical thinking," reminiscent of the podcast's namesake The Hitchhiker's Guide to the Galaxy by Douglas Adams."""
        }
    ]
}

In [ ]:
import pathlib
import json
with pathlib.Path("./ep_json/wiki.json").open("w") as f:
    json.dump(wiki, f)